In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_california_housing

## 문제1

### data load

In [2]:
data=fetch_california_housing()

In [3]:
x=data.data
y=data.target

In [6]:
x.shape,y.shape

((20640, 8), (20640,))

In [9]:
pd.DataFrame(x,columns=data.feature_names).head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [11]:
pd.DataFrame(y,columns=data.target_names).head()

,MedHouseVal
0,4.526
1,3.585
2,3.521
3,3.413
4,3.422


### 전처리

In [13]:
print(data.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

    :Number of Instances: 20640

    :Number of Attributes: 8 numeric, predictive attributes and the target

    :Attribute Information:
        - MedInc        median income in block group
        - HouseAge      median house age in block group
        - AveRooms      average number of rooms per household
        - AveBedrms     average number of bedrooms per household
        - Population    block group population
        - AveOccup      average number of household members
        - Latitude      block group latitude
        - Longitude     block group longitude

    :Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived

In [51]:
pd.DataFrame(x,columns=data.feature_names).describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


In [52]:
pd.DataFrame(x,columns=data.feature_names).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB


In [15]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Dense,Flatten
from sklearn.model_selection import train_test_split

In [16]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=2022,test_size=0.2)

In [18]:
x_train.shape,y_train.shape

((16512, 8), (16512,))

In [53]:
from sklearn.preprocessing import MinMaxScaler

In [54]:
scaler=MinMaxScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

### 모델링

In [55]:
keras.backend.clear_session()
model=keras.models.Sequential()
model.add(Input(shape=(8,)))
model.add(Dense(16,activation='relu'))
model.add(Dense(32,activation='relu'))
model.add(Dense(64,activation='relu'))
model.add(Dense(128,activation='relu'))
model.add(Dense(256,activation='relu'))
model.add(Dense(1))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 16)                144       
                                                                 
 dense_1 (Dense)             (None, 32)                544       
                                                                 
 dense_2 (Dense)             (None, 64)                2112      
                                                                 
 dense_3 (Dense)             (None, 128)               8320      
                                                                 
 dense_4 (Dense)             (None, 256)               33024     
                                                                 
 dense_5 (Dense)             (None, 1)                 257       
                                                                 
Total params: 44,401
Trainable params: 44,401
Non-traina

In [56]:
model.compile(loss='mse',optimizer="adam")

In [46]:
from tensorflow.keras.callbacks import EarlyStopping

In [47]:
es=EarlyStopping(monitor='val_loss',verbose=1,min_delta=0,patience=7,restore_best_weights=True)

In [57]:
model.fit(x_train,y_train,verbose=1,validation_split=0.3,callbacks=[es],epochs=500)

Epoch 1/500
362/362 [==============================] - 2s 3ms/step - loss: 0.8403 - val_loss: 0.5637
Epoch 2/500
362/362 [==============================] - 1s 3ms/step - loss: 0.5409 - val_loss: 0.5694
Epoch 3/500
362/362 [==============================] - 1s 3ms/step - loss: 0.4982 - val_loss: 0.5001
Epoch 4/500
362/362 [==============================] - 1s 3ms/step - loss: 0.4708 - val_loss: 0.4774
Epoch 5/500
362/362 [==============================] - 1s 3ms/step - loss: 0.4562 - val_loss: 0.4712
Epoch 6/500
362/362 [==============================] - 1s 3ms/step - loss: 0.4386 - val_loss: 0.4356
Epoch 7/500
362/362 [==============================] - 1s 3ms/step - loss: 0.4251 - val_loss: 0.4685
Epoch 8/500
362/362 [==============================] - 1s 3ms/step - loss: 0.4090 - val_loss: 0.4090
Epoch 9/500
362/362 [==============================] - 1s 3ms/step - loss: 0.4030 - val_loss: 0.4090
Epoch 10/500
362/362 [==============================] - 1s 3ms/step - loss: 0.3904 - val_lo

In [58]:
pred=model.predict(x_test)

In [59]:
pred[:5].reshape(-1)

array([4.7036057, 1.4990048, 1.3539822, 0.9399331, 2.8430917],
      dtype=float32)

In [60]:
y_test[:5]

array([4.771, 1.371, 1.233, 0.938, 3.26 ])

## 문제2

### data load

In [61]:
from sklearn.datasets import load_wine

In [62]:
wine=load_wine()

In [64]:
print(wine.DESCR)

.. _wine_dataset:

Wine recognition dataset
------------------------

**Data Set Characteristics:**

    :Number of Instances: 178 (50 in each of three classes)
    :Number of Attributes: 13 numeric, predictive attributes and the class
    :Attribute Information:
 		- Alcohol
 		- Malic acid
 		- Ash
		- Alcalinity of ash  
 		- Magnesium
		- Total phenols
 		- Flavanoids
 		- Nonflavanoid phenols
 		- Proanthocyanins
		- Color intensity
 		- Hue
 		- OD280/OD315 of diluted wines
 		- Proline

    - class:
            - class_0
            - class_1
            - class_2
		
    :Summary Statistics:
    
    ============================= ==== ===== ======= =====
                                   Min   Max   Mean     SD
    ============================= ==== ===== ======= =====
    Alcohol:                      11.0  14.8    13.0   0.8
    Malic Acid:                   0.74  5.80    2.34  1.12
    Ash:                          1.36  3.23    2.36  0.27
    Alcalinity of Ash:            1

In [78]:
x=wine.data
y=wine.target

In [79]:
pd.DataFrame(x,columns=wine.feature_names).head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [80]:
pd.DataFrame(x,columns=wine.feature_names).describe()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
count,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000
mean,13.000618,2.336348,2.366517,19.494944,99.741573,2.295112,2.029270,0.361854,1.590899,5.058090,0.957449,2.611685,746.893258
std,0.811827,1.117146,0.274344,3.339564,14.282484,0.625851,0.998859,0.124453,0.572359,2.318286,0.228572,0.709990,314.907474
min,11.030000,0.740000,1.360000,10.600000,70.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000
25%,12.362500,1.602500,2.210000,17.200000,88.000000,1.742500,1.205000,0.270000,1.250000,3.220000,0.782500,1.937500,500.500000
50%,13.050000,1.865000,2.360000,19.500000,98.000000,2.355000,2.135000,0.340000,1.555000,4.690000,0.965000,2.780000,673.500000
75%,13.677500,3.082500,2.557500,21.500000,107.000000,2.800000,2.875000,0.437500,1.950000,6.200000,1.120000,3.170000,985.000000
max,14.830000,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,1680.000000


In [81]:
x.shape,y.shape

((178, 13), (178,))

### 전처리

In [82]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=2022,test_size=0.2)

In [75]:
from tensorflow.keras.utils import to_categorical

In [84]:
scaler=MinMaxScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [85]:
y_train = to_categorical(y_train, 3)
y_test = to_categorical(y_test, 3)

### 모델링

In [101]:
keras.backend.clear_session()
il=keras.layers.Input(shape=(13,))
hl=keras.layers.Dense(8,activation='relu')(il)
hl=keras.layers.Dense(16,activation='relu')(hl)
hl=keras.layers.Dense(32,activation='relu')(hl)
ol=keras.layers.Dense(3,activation='sigmoid')(il)
model=keras.models.Model(inputs=il,outputs=ol)
model.compile(metrics=['accuracy'],optimizer='adam',loss=keras.losses.categorical_crossentropy)

In [102]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 13)]              0         
                                                                 
 dense_3 (Dense)             (None, 3)                 42        
                                                                 
Total params: 42
Trainable params: 42
Non-trainable params: 0
_________________________________________________________________


In [103]:
es=EarlyStopping(verbose=1,patience=5,min_delta=0,restore_best_weights=True,monitor='val_loss')

In [104]:
model.fit(x_train,y_train,verbose=1,callbacks=[es],validation_split=0.1,epochs=100)

Epoch 1/100
4/4 [==============================] - 0s 53ms/step - loss: 1.2761 - accuracy: 0.3622 - val_loss: 1.1383 - val_accuracy: 0.5333
Epoch 2/100
4/4 [==============================] - 0s 9ms/step - loss: 1.2620 - accuracy: 0.3780 - val_loss: 1.1248 - val_accuracy: 0.5333
Epoch 3/100
4/4 [==============================] - 0s 9ms/step - loss: 1.2481 - accuracy: 0.3937 - val_loss: 1.1120 - val_accuracy: 0.5333
Epoch 4/100
4/4 [==============================] - 0s 9ms/step - loss: 1.2348 - accuracy: 0.4016 - val_loss: 1.0997 - val_accuracy: 0.5333
Epoch 5/100
4/4 [==============================] - 0s 10ms/step - loss: 1.2225 - accuracy: 0.4252 - val_loss: 1.0874 - val_accuracy: 0.6000
Epoch 6/100
4/4 [==============================] - 0s 9ms/step - loss: 1.2107 - accuracy: 0.4331 - val_loss: 1.0752 - val_accuracy: 0.6000
Epoch 7/100
4/4 [==============================] - 0s 9ms/step - loss: 1.1992 - accuracy: 0.4409 - val_loss: 1.0632 - val_accuracy: 0.6000
Epoch 8/100
4/4 [========

In [105]:
performance_test = model.evaluate(x_test, y_test)

print('Test Loss : {:.6f},  Test Accuracy : {:.3f}%'.format(performance_test[0], performance_test[1]*100))

2/2 [==============================] - 0s 4ms/step - loss: 462.5774 - accuracy: 0.3056
Test Loss : 462.577362,  Test Accuracy : 30.556%
